# XGBoost e LightGBM

**Objetivo:** usar o `HistGradientBoostingClassifier` do scikit-learn — o mesmo estilo de boosting por histogramas do LightGBM, disponível sem instalar nada — para ver *early stopping* e regularização, e comparar (opcionalmente) com o XGBoost de verdade.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
import time

dados = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(dados.data, dados.target,
                                          test_size=0.3, random_state=SEMENTE, stratify=dados.target)
print("treino:", X_tr.shape)

## 1. Early stopping: quantas árvores bastam?

Com `early_stopping=True`, o modelo separa uma fração de validação e **para sozinho** quando o erro de validação deixa de melhorar — em vez de fixar o número de árvores no chute.

In [ ]:
inicio = time.time()
modelo = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=500,
                                        early_stopping=True, validation_fraction=0.2,
                                        n_iter_no_change=15, random_state=SEMENTE)
modelo.fit(X_tr, y_tr)
print("acuracia no teste:", round(modelo.score(X_te, y_te), 3))
print("arvores efetivamente usadas (parou sozinho):", modelo.n_iter_, "de 500 possiveis")
print("tempo de treino:", round(time.time() - inicio, 2), "s")

## 2. Regularização via profundidade das folhas

Limitar `max_leaf_nodes` regulariza (árvores menores, menos overfitting). Varremos alguns valores e comparamos treino × teste.

In [ ]:
for folhas in [3, 7, 15, 31, 63]:
    m = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=200,
                                       max_leaf_nodes=folhas, random_state=SEMENTE)
    m.fit(X_tr, y_tr)
    print("max_leaf_nodes", str(folhas).rjust(2),
          "| treino", round(m.score(X_tr, y_tr), 3),
          "| teste", round(m.score(X_te, y_te), 3))

## 3. (Opcional) XGBoost de verdade

Se o `xgboost` estiver disponível (no Colab, costuma estar), a célula abaixo o treina e compara. Se não estiver, ela avisa e segue — o resto do notebook não depende dele.

In [ ]:
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=3,
                        reg_lambda=1.0, eval_metric="logloss", random_state=SEMENTE)
    xgb.fit(X_tr, y_tr)
    print("acuracia XGBoost:", round(xgb.score(X_te, y_te), 3))
except Exception as erro:
    print("xgboost nao disponivel neste ambiente — pulando.")
    print("para instalar no Colab: !pip install xgboost")
    print("detalhe:", type(erro).__name__)

## Exercício

Na varredura do item 2, o que acontece com a diferença entre acurácia de treino e de teste conforme `max_leaf_nodes` aumenta? Como isso se relaciona com a regularização?

<details><summary>Ver resposta</summary>

Neste conjunto a acurácia de **treino** já satura em ~100% mesmo com poucas folhas; o que muda é o **teste**, melhor com folhas intermediárias (7–15) e que **cai um pouco** para árvores maiores. Ou seja, capacidade extra não ajuda o teste e pode piorá-lo (overfitting): o *gap* entre treino e teste só aumenta. Limitar as folhas é **regularizar** — árvores menores generalizam melhor, exatamente o papel do termo de penalidade do XGBoost.

</details>